# GI/GI/N(+GI) Queue Simulation Package Walkthrough

This notebook provides a guided introduction to the queue simulation package.

The goals are to:

• demonstrate how to construct queueing systems  
• run simulations  
• compute performance metrics  
• compare scheduling policies  
• extend from single-class to multiclass systems  

The simulator implements event-driven GI/GI/N queues with optional abandonment.

Supported features include:

- arbitrary interarrival distributions
- arbitrary service distributions
- arbitrary patience distributions
- multiple service policies
- heterogeneous customer classes

The package is designed for **queueing research and policy experiments**, including
the TIQ and mTIQ policies studied in Bassamboo et al.

This notebook walks through the basic workflow.

## Package Structure

The package is organized into several modules:

core/
: event engine and system state

dists/
: probability distributions used by arrivals, service, and patience

policies/
: scheduling rules (FCFS, LCFS, TIQ, priority policies, mTIQ)

metrics/
: performance metrics (queue length, abandonment rates, etc.)

outputs/
: tools for converting simulation results to tables

fluid/
: fluid approximations used to compute TIQ thresholds

Notebooks in this repository use these modules to construct experiments.

In [1]:
import numpy as np
import pandas as pd

from gi_gi_n_gi_multiclass.dists.common import Exponential, Weibull, LogNormal

from gi_gi_n_gi_multiclass.core.types import SimConfig
from gi_gi_n_gi_multiclass.core.engine import run_sim

from gi_gi_n_gi_multiclass.metrics.basic import summary_table

from gi_gi_n_gi_multiclass.policies.fcfs import FCFS

## Example 1: A Simple GI/GI/N Queue

We begin with a single-class system.

System parameters:

Arrival rate: λ = 8  
Service rate: μ = 1  
Servers: N = 10  

Since λ < Nμ, the system is **stable** and should experience little congestion.

In [16]:
cfg = SimConfig(
    n_servers=10,
    interarrival=Exponential(rate=8.0),
    service=Exponential(rate=1.0),
    patience=None,
    policy=FCFS(),
    seed=1,
    run_time=2000,
    warmup_time=500
)

res = run_sim(cfg)

summary_table(res)

,policy,n_servers,arrivals,served,abandoned,lambda_hat,served_rate,abandon_rate,W_mean,service_mean,patience_mean,L_timeavg,Lq_timeavg,end_time
0,FCFS,10,15981,15981,0,7.9905,7.9905,0.0,1.196857,1.014921,NaN,9.570884,1.45735,2500


### Interpreting the Output

Key metrics:

λ̂
: estimated arrival rate

served_rate
: rate of completed services

W_mean
: mean time in system for served customers

L_timeavg
: average number of customers in system

Lq_timeavg
: average queue length

These correspond to standard queueing quantities such as

Little's Law:

L ≈ λW

## Changing the Arrival Distribution

The simulator supports general distributions.

For example we can replace exponential arrivals with a Weibull distribution.

In [17]:
cfg_weibull = SimConfig(
    n_servers=10,
    interarrival=Weibull(shape=1.5, scale=1/8),
    service=Exponential(rate=1.0),
    patience=None,
    policy=FCFS(),
    seed=1,
    run_time=2000,
    warmup_time=500
)

res = run_sim(cfg_weibull)
summary_table(res)

,policy,n_servers,arrivals,served,abandoned,lambda_hat,served_rate,abandon_rate,W_mean,service_mean,patience_mean,L_timeavg,Lq_timeavg,end_time
0,FCFS,10,17698,17696,0,8.849,8.848,0.0,1.344661,1.015952,NaN,11.902837,2.909503,2500


## Adding Customer Abandonment

Customers may leave the system if their patience expires.

This converts the system from

GI/GI/N

to

GI/GI/N+GI

In [31]:
cfg = SimConfig(
    n_servers=10,
    interarrival=Exponential(rate=5.0),
    service=Exponential(rate=0.5),
    patience=Exponential(rate=5.0),
    policy=FCFS(),
    seed=1,
    run_time=2000,
    warmup_time=500
)
res = run_sim(cfg)

summary_table(res)

,policy,n_servers,arrivals,served,abandoned,lambda_hat,served_rate,abandon_rate,W_mean,service_mean,patience_mean,L_timeavg,Lq_timeavg,end_time
0,FCFS,10,9945,8184,1760,4.9725,4.092,0.88,2.011674,2.016116,0.198096,8.322949,0.179018,2500


## Changing the Scheduling Policy

Scheduling rules determine which waiting customer receives service.

Examples implemented in the package include:

FCFS  
LCFS  
priority rules  
TIQ policies

In [32]:
from gi_gi_n_gi_multiclass.policies.lcfs import LCFS

cfg = SimConfig(
    n_servers=10,
    interarrival=Exponential(rate=5.0),
    service=Exponential(rate=0.5),
    patience=Exponential(rate=5.0),
    policy=LCFS(),
    seed=1,
    run_time=2000,
    warmup_time=500
)

res = run_sim(cfg)

summary_table(res)

,policy,n_servers,arrivals,served,abandoned,lambda_hat,served_rate,abandon_rate,W_mean,service_mean,patience_mean,L_timeavg,Lq_timeavg,end_time
0,LCFS,10,9945,8134,1810,4.9725,4.067,0.905,2.021421,2.016116,0.198096,8.331057,0.183048,2500


## Moving to Multiclass Systems

Many queueing problems involve heterogeneous customers.

The simulator supports multiple customer classes, each with their own:

- arrival process
- service distribution
- patience distribution

In [33]:
from gi_gi_n_gi_multiclass.core.types_multi import CustomerClass, MultiSimConfig
from gi_gi_n_gi_multiclass.core.engine_multi import run_sim_multi

from gi_gi_n_gi_multiclass.policies.priority_fcfs import PriorityFCFS
from gi_gi_n_gi_multiclass.outputs.tables import class_level_summary

In [34]:
class0 = CustomerClass(
    name="class0",
    interarrival=Exponential(rate=4),
    service=Exponential(rate=1),
    patience=Exponential(rate=0.5)
)

class1 = CustomerClass(
    name="class1",
    interarrival=Exponential(rate=6),
    service=Exponential(rate=1),
    patience=LogNormal(meanlog=0, sdlog=1)
)

classes = [class0, class1]

In [36]:
cfg = MultiSimConfig(
    n_servers=10,
    classes=classes,
    policy=PriorityFCFS(priority_order=[1,0]),
    seed=1,
    run_time=2000,
    warmup_time=500
)

res = run_sim_multi(cfg)

class_level_summary(res)

,scope,arrivals,served,abandoned,lambda_hat,served_rate,abandon_rate,Wq_mean_served,W_mean_served,L_timeavg,Lq_timeavg
0,OVERALL,19939,17846,2084,9.9695,8.9230,1.042,0.191541,1.196462,11.192882,2.215995
1,class_0,7926,6447,1478,3.9630,3.2235,0.739,0.328422,1.340191,4.743021,1.481487
2,class_1,12013,11399,606,6.0065,5.6995,0.303,0.114124,1.115172,6.442896,0.734121


### Interpreting Multiclass Output

The summary table now includes statistics for:

• the overall system  
• each customer class

This allows evaluation of policies that trade off performance across classes.

## Summary

This notebook demonstrated how to:

• define queueing systems  
• run simulations  
• compute system performance metrics  
• change scheduling policies  
• simulate heterogeneous customer classes  

The following notebooks explore specific research questions:

Single-class systems:
- policy ranking under different hazard shapes
- optimal TIQ thresholds

Multiclass systems:
- validation of the βᵢ index rule
- cross-class mTIQ policies
- within-class TIQ improvements